In [1]:
print("Hello World")

Hello World


In [2]:
%pip install -U transformers pillow numpy matplotlib tqdm scikit-learn

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import gc
import numpy as np
import matplotlib.pyplot as plt

import torch
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader

from PIL import Image
from tqdm.auto import tqdm

from transformers import CLIPModel, CLIPProcessor

c:\Users\smitp\.conda\envs\Medical\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("CUDA:", torch.version.cuda)

DEVICE = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using:", DEVICE)

PyTorch: 2.14.1+cu130
CUDA available: True
GPU: NVIDIA GeForce RTX 5050 Laptop GPU
CUDA: 13.0
Using: cuda


In [8]:
DATASET_ROOT = r"C:\AI\Medical\Liver\data\stage1_balanced"

TRAIN_DIR = os.path.join(DATASET_ROOT, "train")
VAL_DIR = os.path.join(DATASET_ROOT, "val")
TEST_DIR = os.path.join(DATASET_ROOT, "test")

print("Train:", TRAIN_DIR)
print("Val:", VAL_DIR)
print("Test:", TEST_DIR)

Train: C:\AI\Medical\Liver\data\stage1_balanced\train
Val: C:\AI\Medical\Liver\data\stage1_balanced\val
Test: C:\AI\Medical\Liver\data\stage1_balanced\test


In [9]:
CLASSES = [
    "Pneumonia",
    "Corona Virus Disease",
    "Normal",
    "Tuberculosis"
]

CLASS_TO_IDX = {
    name: i
    for i, name in enumerate(CLASSES)
}

IDX_TO_CLASS = {
    i: name
    for i, name in enumerate(CLASSES)
}

NUM_CLASSES = len(CLASSES)

print(CLASS_TO_IDX)

{'Pneumonia': 0, 'Corona Virus Disease': 1, 'Normal': 2, 'Tuberculosis': 3}


In [10]:
for split_name, split_dir in [
    ("TRAIN", TRAIN_DIR),
    ("VAL", VAL_DIR),
    ("TEST", TEST_DIR)
]:

    print("\n" + "=" * 55)
    print(split_name)
    print("=" * 55)

    total = 0

    for class_name in CLASSES:

        class_dir = os.path.join(
            split_dir,
            class_name
        )

        files = [
            f for f in os.listdir(class_dir)
            if f.lower().endswith(
                (".jpg", ".jpeg", ".png", ".bmp")
            )
        ]

        print(
            f"{class_name}: {len(files)}"
        )

        total += len(files)

    print("TOTAL:", total)


TRAIN
Pneumonia: 1204
Corona Virus Disease: 1218
Normal: 1207
Tuberculosis: 1220
TOTAL: 4849

VAL
Pneumonia: 400
Corona Virus Disease: 406
Normal: 402
Tuberculosis: 406
TOTAL: 1614

TEST
Pneumonia: 402
Corona Virus Disease: 407
Normal: 404
Tuberculosis: 408
TOTAL: 1621


In [11]:
MODEL_NAME = "openai/clip-vit-base-patch32"

processor = CLIPProcessor.from_pretrained(
    MODEL_NAME
)

model = CLIPModel.from_pretrained(
    MODEL_NAME
)

model = model.to(DEVICE)

print("CLIP loaded.")

Loading weights: 100%|██████████| 398/398 [00:00<00:00, 25440.56it/s]


CLIP loaded.


In [12]:
for parameter in model.parameters():
    parameter.requires_grad = True

trainable_params = sum(
    p.numel()
    for p in model.parameters()
    if p.requires_grad
)

total_params = sum(
    p.numel()
    for p in model.parameters()
)

print(
    f"Trainable parameters: "
    f"{trainable_params:,}"
)

print(
    f"Total parameters: "
    f"{total_params:,}"
)

Trainable parameters: 151,277,313
Total parameters: 151,277,313


In [19]:
TEXT_TEMPLATES = {

    "Corona Virus Disease": [
        "a chest X-ray showing COVID-19",
        "a chest radiograph of COVID-19",
        "a lung X-ray showing coronavirus disease",
        "a chest X-ray with COVID-19 related lung abnormalities",
        "a chest radiograph of a patient with COVID-19",

        "a chest X-ray indicating COVID-19",
        "a chest radiograph associated with coronavirus disease",
        "an X-ray image of COVID-19 pneumonia",
        "a lung radiograph showing COVID-19 related abnormalities",
        "a chest X-ray from a patient with COVID-19",

        "a radiograph of a patient with coronavirus disease",
        "a chest image showing COVID-19 related pneumonia",
        "a chest X-ray with findings associated with COVID-19",
        "a medical chest radiograph showing coronavirus disease",
        "a chest radiograph from a patient with COVID-19 lung involvement"
    ],

    "Normal": [
        "a normal chest X-ray",
        "a chest radiograph with clear lungs",
        "a normal lung X-ray",
        "a healthy chest radiograph",
        "a chest X-ray showing no obvious lung abnormality",

        "a chest radiograph showing normal lungs",
        "an X-ray of a normal chest",
        "a normal chest radiograph",
        "a clear lung radiograph",
        "a chest X-ray with no apparent pulmonary abnormality",

        "a medical chest X-ray showing normal findings",
        "a chest image with clear lung fields",
        "a radiograph of a healthy chest",
        "a chest radiograph without apparent abnormalities",
        "a normal lung examination on chest X-ray"
    ],

    "Pneumonia": [
        "a chest X-ray showing pneumonia",
        "a chest radiograph of pneumonia",
        "a lung X-ray with pneumonia",
        "a chest radiograph demonstrating pneumonia",
        "a chest X-ray of a patient with pneumonia",

        "a chest X-ray indicating pneumonia",
        "a chest radiograph consistent with pneumonia",
        "an X-ray image of pneumonia",
        "a lung radiograph showing pneumonia",
        "a chest X-ray associated with pneumonia",

        "a radiograph of a patient with pneumonia",
        "a chest image showing pneumonia",
        "a chest X-ray with findings associated with pneumonia",
        "a medical chest radiograph showing pneumonia",
        "a chest radiograph from a patient with pneumonia"
    ],

    "Tuberculosis": [
        "a chest X-ray showing tuberculosis",
        "a chest radiograph of pulmonary tuberculosis",
        "a lung X-ray with tuberculosis",
        "a chest radiograph demonstrating tuberculosis",
        "a chest X-ray of a patient with pulmonary tuberculosis",

        "a chest X-ray indicating tuberculosis",
        "a chest radiograph consistent with pulmonary tuberculosis",
        "an X-ray image of tuberculosis",
        "a lung radiograph showing tuberculosis",
        "a chest X-ray associated with tuberculosis",

        "a radiograph of a patient with tuberculosis",
        "a chest image showing pulmonary TB",
        "a chest X-ray with findings associated with tuberculosis",
        "a medical chest radiograph showing pulmonary tuberculosis",
        "a chest radiograph from a patient with tuberculosis"
    ]
}

In [20]:
class LungDataset(Dataset):

    def __init__(self, root_dir):

        self.samples = []

        for class_name in CLASSES:

            class_dir = os.path.join(
                root_dir,
                class_name
            )

            for filename in os.listdir(class_dir):

                if filename.lower().endswith(
                    (".jpg", ".jpeg", ".png", ".bmp")
                ):

                    self.samples.append({
                        "path": os.path.join(
                            class_dir,
                            filename
                        ),
                        "label": CLASS_TO_IDX[
                            class_name
                        ],
                        "class_name": class_name
                    })

        print(
            os.path.basename(root_dir),
            "images:",
            len(self.samples)
        )

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):

        sample = self.samples[index]

        image = Image.open(
            sample["path"]
        ).convert("RGB")

        return (
            image,
            sample["label"]
        )

In [21]:
train_dataset = LungDataset(
    TRAIN_DIR
)

val_dataset = LungDataset(
    VAL_DIR
)

test_dataset = LungDataset(
    TEST_DIR
)

train images: 4849
val images: 1614
test images: 1621


In [22]:
def collate_fn(batch):

    images = [
        item[0]
        for item in batch
    ]

    labels = torch.tensor(
        [
            item[1]
            for item in batch
        ],
        dtype=torch.long
    )

    inputs = processor(
        images=images,
        return_tensors="pt"
    )

    inputs["labels"] = labels

    return inputs

In [23]:
BATCH_SIZE = 4

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_fn
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_fn
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    collate_fn=collate_fn
)

In [25]:
ALL_PROMPTS = []
PROMPT_LABELS = []

for class_name in CLASSES:

    for prompt in TEXT_TEMPLATES[class_name]:

        ALL_PROMPTS.append(prompt)

        PROMPT_LABELS.append(
            CLASS_TO_IDX[class_name]
        )

print(
    "Total prompts:",
    len(ALL_PROMPTS)
)

print(ALL_PROMPTS)

Total prompts: 60
['a chest X-ray showing pneumonia', 'a chest radiograph of pneumonia', 'a lung X-ray with pneumonia', 'a chest radiograph demonstrating pneumonia', 'a chest X-ray of a patient with pneumonia', 'a chest X-ray indicating pneumonia', 'a chest radiograph consistent with pneumonia', 'an X-ray image of pneumonia', 'a lung radiograph showing pneumonia', 'a chest X-ray associated with pneumonia', 'a radiograph of a patient with pneumonia', 'a chest image showing pneumonia', 'a chest X-ray with findings associated with pneumonia', 'a medical chest radiograph showing pneumonia', 'a chest radiograph from a patient with pneumonia', 'a chest X-ray showing COVID-19', 'a chest radiograph of COVID-19', 'a lung X-ray showing coronavirus disease', 'a chest X-ray with COVID-19 related lung abnormalities', 'a chest radiograph of a patient with COVID-19', 'a chest X-ray indicating COVID-19', 'a chest radiograph associated with coronavirus disease', 'an X-ray image of COVID-19 pneumonia', 

In [26]:
prompt_inputs = processor(
    text=ALL_PROMPTS,
    return_tensors="pt",
    padding=True,
    truncation=True
)

PROMPT_INPUT_IDS = prompt_inputs[
    "input_ids"
].to(DEVICE)

PROMPT_ATTENTION_MASK = prompt_inputs[
    "attention_mask"
].to(DEVICE)

PROMPT_LABELS = torch.tensor(
    PROMPT_LABELS,
    dtype=torch.long,
    device=DEVICE
)

print(
    "Prompt tokens:",
    PROMPT_INPUT_IDS.shape
)

Prompt tokens: torch.Size([60, 18])


In [27]:
def encode_images(pixel_values):

    vision_outputs = model.vision_model(
        pixel_values=pixel_values
    )

    image_embeds = model.visual_projection(
        vision_outputs.pooler_output
    )

    image_embeds = F.normalize(
        image_embeds,
        dim=-1
    )

    return image_embeds

In [28]:
def encode_all_prompts():

    text_outputs = model.text_model(
        input_ids=PROMPT_INPUT_IDS,
        attention_mask=PROMPT_ATTENTION_MASK
    )

    text_embeds = model.text_projection(
        text_outputs.pooler_output
    )

    text_embeds = F.normalize(
        text_embeds,
        dim=-1
    )

    return text_embeds

In [29]:
def get_class_text_embeddings():

    text_embeds = encode_all_prompts()

    class_embeddings = []

    for class_idx in range(NUM_CLASSES):

        mask = (
            PROMPT_LABELS ==
            class_idx
        )

        class_embed = text_embeds[mask].mean(
            dim=0
        )

        class_embed = F.normalize(
            class_embed,
            dim=0
        )

        class_embeddings.append(
            class_embed
        )

    return torch.stack(
        class_embeddings
    )

In [30]:
batch = next(
    iter(train_loader)
)

pixel_values = batch[
    "pixel_values"
].to(DEVICE)

labels = batch[
    "labels"
].to(DEVICE)

with torch.no_grad():

    image_embeds = encode_images(
        pixel_values
    )

    class_text_embeds = (
        get_class_text_embeddings()
    )

print(
    "Image embeddings:",
    image_embeds.shape
)

print(
    "Class text embeddings:",
    class_text_embeds.shape
)

Image embeddings: torch.Size([4, 512])
Class text embeddings: torch.Size([4, 512])


In [31]:
with torch.no_grad():

    test_logits = (
        model.logit_scale.exp()
        *
        image_embeds
        @
        class_text_embeds.T
    )

print(
    "Logits:",
    test_logits.shape
)

print(
    "Predictions:",
    test_logits.argmax(dim=1)
)

print(
    "Labels:",
    labels
)

Logits: torch.Size([4, 4])
Predictions: tensor([0, 1, 1, 0], device='cuda:0')
Labels: tensor([3, 2, 0, 3], device='cuda:0')


In [32]:
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=1e-6,
    weight_decay=1e-4
)

print("Optimizer ready.")

Optimizer ready.


In [33]:
USE_AMP = (
    DEVICE.type == "cuda"
)

scaler = torch.amp.GradScaler(
    "cuda",
    enabled=USE_AMP
)

print(
    "AMP enabled:",
    USE_AMP
)

AMP enabled: True


In [34]:
def train_one_epoch():

    model.train()

    running_loss = 0.0
    correct = 0
    total = 0

    progress = tqdm(
        train_loader,
        desc="Training"
    )

    for batch in progress:

        pixel_values = batch[
            "pixel_values"
        ].to(
            DEVICE,
            non_blocking=True
        )

        labels = batch[
            "labels"
        ].to(
            DEVICE,
            non_blocking=True
        )

        optimizer.zero_grad(
            set_to_none=True
        )

        with torch.autocast(
            device_type=DEVICE.type,
            dtype=torch.float16,
            enabled=USE_AMP
        ):

            # -------------------------
            # IMAGE EMBEDDINGS
            # -------------------------

            image_embeds = encode_images(
                pixel_values
            )

            # -------------------------
            # TEXT EMBEDDINGS
            # -------------------------

            class_text_embeds = (
                get_class_text_embeddings()
            )

            # -------------------------
            # SIMILARITY
            # -------------------------

            logit_scale = (
                model.logit_scale.exp()
            )

            logits = (
                logit_scale *
                image_embeds @
                class_text_embeds.T
            )

            # -------------------------
            # CLASSIFICATION LOSS
            # -------------------------

            loss = F.cross_entropy(
                logits.float(),
                labels
            )

        # -------------------------
        # BACKPROP
        # -------------------------

        scaler.scale(
            loss
        ).backward()

        scaler.unscale_(
            optimizer
        )

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            max_norm=1.0
        )

        scaler.step(
            optimizer
        )

        scaler.update()

        # -------------------------
        # METRICS
        # -------------------------

        predictions = logits.argmax(
            dim=1
        )

        batch_correct = (
            predictions == labels
        ).sum().item()

        batch_size = labels.size(0)

        correct += batch_correct
        total += batch_size

        running_loss += (
            loss.item() *
            batch_size
        )

        progress.set_postfix(
            loss=f"{loss.item():.4f}",
            acc=f"{correct / total:.4f}"
        )

        del pixel_values
        del labels
        del image_embeds
        del class_text_embeds
        del logits

    return (
        running_loss / total,
        correct / total
    )

In [35]:
def validate():

    model.eval()

    running_loss = 0.0
    correct = 0
    total = 0

    with torch.inference_mode():

        class_text_embeds = (
            get_class_text_embeddings()
        )

        progress = tqdm(
            val_loader,
            desc="Validation"
        )

        for batch in progress:

            pixel_values = batch[
                "pixel_values"
            ].to(DEVICE)

            labels = batch[
                "labels"
            ].to(DEVICE)

            with torch.autocast(
                device_type=DEVICE.type,
                dtype=torch.float16,
                enabled=USE_AMP
            ):

                image_embeds = (
                    encode_images(
                        pixel_values
                    )
                )

                logits = (
                    model.logit_scale.exp()
                    *
                    image_embeds
                    @
                    class_text_embeds.T
                )

                loss = F.cross_entropy(
                    logits.float(),
                    labels
                )

            predictions = logits.argmax(
                dim=1
            )

            batch_size = labels.size(0)

            running_loss += (
                loss.item() *
                batch_size
            )

            correct += (
                predictions == labels
            ).sum().item()

            total += batch_size

    return (
        running_loss / total,
        correct / total
    )